In [18]:
import os
from dotenv import load_dotenv
load_dotenv()
#hf_token = os.getenv("HF_TOKEN")
#print(hf_token)
#print(os.getenv("HF_HOME"))

True

Completely Offline Deployment\
Once downloaded, you can run without internet:

In [19]:
from transformers import AutoModel
model = AutoModel.from_pretrained("openai/privacy-filter")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

[transformers] OpenAIPrivacyFilterModel LOAD REPORT from: openai/privacy-filter
Key          | Status     |  | 
-------------+------------+--+-
score.weight | UNEXPECTED |  | 
score.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "openai/privacy-filter",
    use_fast=False
)

Check if model is cached

In [ ]:
from huggingface_hub import scan_cache_dir

print(scan_cache_dir())

Load model from local cache (offline-safe)

In [21]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

MODEL_PATH = "openai/privacy-filter"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    use_fast=False   # important fallback
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

privacy_pipe = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

Run inference

In [ ]:
REDACTION_MAP = {
    "private_person": "[REDACTED_NAME]",
    "private_email": "[REDACTED_EMAIL]",
    "private_phone": "[REDACTED_PHONE]",
    "private_location": "[REDACTED_LOCATION]",
    "private_address": "[REDACTED_ADDRESS]",
    "private_date": "[REDACTED_DATE]",
    "private_credit_card": "[REDACTED_CREDIT_CARD]",
    "private_ssn": "[REDACTED_SSN]",
    "private_ip": "[REDACTED_IP]",
    "private_url": "[REDACTED_URL]",
    "private_bank_account": "[REDACTED_BANK_ACCOUNT]",
}


def merge_entities(entities):
    merged = []
    for e in sorted(entities, key=lambda x: (x["start"], x["end"])):
        if not merged:
            merged.append(dict(e))
            continue

        last = merged[-1]
        if e.get("entity_group") == last.get("entity_group") and e["start"] <= last["end"]:
            last["end"] = max(last["end"], e["end"])
            if e.get("score", 0) > last.get("score", 0):
                last["score"] = e["score"]
            continue

        merged.append(dict(e))

    return merged


def redact(text, entities):
    entities = merge_entities(entities)
    output = []
    cursor = 0

    for e in entities:
        start = e["start"]
        end = e["end"]
        label = e.get("entity_group")

        if start < cursor:
            continue

        output.append(text[cursor:start])

        entity_text = text[start:end]
        leading_whitespace_len = len(entity_text) - len(entity_text.lstrip())
        if leading_whitespace_len:
            output.append(entity_text[:leading_whitespace_len])

        output.append(REDACTION_MAP.get(label, "[REDACTED]"))
        cursor = end

    output.append(text[cursor:])
    return "".join(output)

In [34]:
text = "My name is John Doe and my email is john.doe@gmail.com"
entities = privacy_pipe(text)
print(redact(text, entities))

My name is [REDACTED_NAME] and my email is [REDACTED_EMAIL]
